In [ ]:
# DSNML: retail sales modelling notebook
from google.colab import drive
import os
import warnings
import numpy as np
import pandas as pd

drive.mount('/content/drive')
folder_path = '/content/drive/MyDrive/DSNML'
os.chdir(folder_path)
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
display(train_df.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
display(train_df.describe())
display(train_df.describe(include='object'))

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
sns.histplot(train_df['total_sales'], bins=50, kde=True, ax=axes[0, 0], color='teal')
axes[0, 0].set_title('Distribution of Total Sales')
sns.heatmap(train_df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm', fmt='.2f', ax=axes[0, 1])
sns.boxplot(data=train_df, x='store_format', y='total_sales', ax=axes[1, 0])
sns.scatterplot(data=train_df, x='product_price', y='total_sales', hue='store_location_tier', alpha=.5, ax=axes[1, 1])
plt.tight_layout()
plt.show()

In [ ]:
!pip install -q catboost lightgbm xgboost shap

from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from catboost import CatBoostRegressor

SEED = 42
TARGET = 'total_sales'
ID_COL = 'id'
CAT_FEATURES = ['product_code', 'fat_content', 'product_category', 'store_code', 'store_size', 'store_location_tier', 'store_format', 'store_tier_format']

def prepare_data(df):
    df = df.copy()
    for col in ['product_category', 'fat_content']:
        if col in df:
            df[col] = df[col].fillna('Unknown').astype(str).str.lower().str.strip()
    if 'product_weight_kg' in df:
        df['product_weight_kg'] = df['product_weight_kg'].fillna(-1)
        df['weight_clean'] = df['product_weight_kg'].replace(-1, np.nan)
        df['price_per_kg'] = df['product_price'] / df['weight_clean']
    if 'store_size' in df:
        df['store_size'] = df['store_size'].fillna('Unknown').astype(str)
    df['store_tier_format'] = df['store_location_tier'].astype(str) + '_' + df['store_format'].astype(str)
    non_edible = ['household', 'health and hygiene', 'hard drinks', 'others']
    df['fat_content_adjusted'] = np.where(df['product_category'].isin(non_edible), 'Non-Edible', df['fat_content'])
    return df

train = prepare_data(train_df)
test = prepare_data(test_df)
for col in CAT_FEATURES:
    if col in train:
        train[col] = train[col].fillna('Unknown').astype(str)
        test[col] = test[col].fillna('Unknown').astype(str)

X = train.drop(columns=[ID_COL, TARGET])
y = train[TARGET]
X_test = test.drop(columns=[ID_COL])
cat_idx = [X.columns.get_loc(c) for c in CAT_FEATURES if c in X.columns]
print('Features:', X.shape[1])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros(len(X))
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
    model = CatBoostRegressor(iterations=1000, depth=6, learning_rate=.05, l2_leaf_reg=3, random_strength=1, cat_features=cat_idx, random_seed=SEED, verbose=False)
    model.fit(X.iloc[tr_idx], y.iloc[tr_idx], eval_set=(X.iloc[va_idx], y.iloc[va_idx]), early_stopping_rounds=50)
    pred = np.clip(model.predict(X.iloc[va_idx]), 0, None)
    oof[va_idx] = pred
    score = root_mean_squared_error(y.iloc[va_idx], pred)
    fold_scores.append(score)
    print(f'Fold {fold} RMSE: {score:.4f}')

print(f'Mean CV RMSE: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}')
train['oof_predictions'] = oof
train['residual'] = y - oof
sns.histplot(train['residual'], bins=50, kde=True)
plt.title('Distribution of Residuals')
plt.show()

In [ ]:
final_model = CatBoostRegressor(iterations=1000, depth=6, learning_rate=.05, l2_leaf_reg=3, random_strength=1, cat_features=cat_idx, random_seed=SEED, verbose=100)
final_model.fit(X, y)
predictions = np.clip(final_model.predict(X_test), 0, None)
submission = pd.DataFrame({ID_COL: test_df[ID_COL], TARGET: predictions})
submission.to_csv('final_catboost_submission.csv', index=False)
print("Saved final_catboost_submission.csv")